# Weather - Lane A (the real experience: you prompt, the agent builds)

**SISMID 2026 - Day 2, 11:00.** You drive a coding agent (Codex, Claude Code, or
Antigravity CLI). Paste each prompt, run the code, apply the check.


## About this data source

**Open-Meteo.** A free weather service built for programs rather than people: give it a
latitude, longitude and date range, and it returns decades of daily weather. No account, no
key, no cost. Think of it as: *"the weather history of any point on Earth, as a table."*

- **Explore it in a browser:** <https://open-meteo.com/en/docs/historical-weather-api>
  (a live form: type a city, tick *temperature* and *dew point*, and it builds the exact web
  address that returns the data)
- Research-grade source behind it (ERA5): <https://cds.climate.copernicus.eu/>

> Why weather at all? **Absolute humidity** is the established driver of influenza
> seasonality (Shaman & Kohn 2009; Shaman et al. 2010, 2012). Weather does not count cases;
> it shifts the conditions for transmission.


## Step 1: pull the weather

> *Using the Open-Meteo historical archive API (free, no key), pull daily mean*
> *temperature and dew point for Atlanta (33.749, -84.388) from 2016 to now. Return a*
> *tidy DataFrame and report the date range.*


In [1]:
# Agent's Open-Meteo pull:
import os, time, json
import urllib.request, urllib.parse
import pandas as pd, matplotlib.pyplot as plt

UA = {"User-Agent": "SISMID2026-course/1.0 (your-email@example.com)"}


def cache_path(fname):
    for p in (f"../data/{fname}", f"data/{fname}", f"./{fname}"):
        if os.path.exists(p):
            return p
    return None


def fetch(url, timeout=120):
    return urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=timeout).read()


MY_PLACE, MY_LAT, MY_LON = "Atlanta", 33.749, -84.388
START = "2016-01-01"


def get_weather():
    """Daily mean temperature + dew point for MY_LAT/MY_LON, 2016 to today, from the
    Open-Meteo historical archive API (free, no key). Falls back to the cache on failure."""
    end = time.strftime("%Y-%m-%d")
    url = ("https://archive-api.open-meteo.com/v1/archive"
           f"?latitude={MY_LAT}&longitude={MY_LON}&start_date={START}&end_date={end}"
           "&daily=temperature_2m_mean,dew_point_2m_mean&timezone=America%2FNew_York")
    try:
        d = json.loads(fetch(url))["daily"]
        return pd.DataFrame({"date": pd.to_datetime(d["time"]),
                             "temp_c": d["temperature_2m_mean"],
                             "dewpoint_c": d["dew_point_2m_mean"]})
    except Exception as e:
        print(f"Live Open-Meteo pull failed ({type(e).__name__}): {e}")
        p = cache_path("openmeteo_atlanta_weather.csv")
        print("Using cached snapshot:", p)
        return pd.read_csv(p, parse_dates=["date"])


wx = get_weather()
print(f"{len(wx)} days, {wx['date'].min().date()} to {wx['date'].max().date()}")
wx.head()


3855 days, 2016-01-01 to 2026-07-21


,date,temp_c,dewpoint_c
0,2016-01-01,7.1,-0.9
1,2016-01-02,4.7,-4.0
2,2016-01-03,5.3,-2.8
3,2016-01-04,3.1,-5.1
4,2016-01-05,0.3,-9.0


## Step 2: compute absolute humidity

> *Compute absolute humidity in g/m^3 from temperature T and dew point Td:*
> *e = 6.112 * exp(17.67*Td/(Td+243.5)); AH = 216.7 * e / (T+273.15). Resample to weekly*
> *and plot it, then plot the mean AH by calendar month.*

**Your check:** does AH bottom out in winter? That is the Shaman result: low absolute
humidity precedes influenza season.


In [2]:
# Agent's absolute humidity + plots:


## Step 3: sanity-check and save

> *Report the date range, missing values, and the AH range, then save a tidy CSV.*


In [3]:
# Agent's checks + save:


## Reflection

- Weather is a **modulator**, not a case count.
- **Stretch:** swap in a tropical city and compare the AH cycle.
